In [5]:
import cv2
import imutils
import time

TrDict={'csrt':cv2.TrackerCSRT_create,
        'kcf':cv2.TrackerKCF_create,
        'mil':cv2.TrackerMIL_create,
        # 'boosting':cv2.TrackerBoosting_create,
        # 'medianflow':cv2.TrackerMedianFlow_create,
        # 'mosse':cv2.TrackerMOSSE_create
        }

In [2]:
tracker=TrDict['csrt']()
# tracker=TrDict['kcf']()
# tracker=TrDict['mil']()
# tracker=TrDict['boosting']()
# tracker=TrDict['medianflow']()
# tracker=TrDict['mosse']()

In [6]:
cap = cv2.VideoCapture(1)

ret, frame = cap.read()

if not ret:
    print("Camera error")
    cap.release()
    exit()

frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
frame = imutils.resize(frame, width=800)

cv2.imshow("Frame", frame)
cv2.waitKey(1)

bbox = cv2.selectROI("Frame", frame)

prev = time.time()

tracker = cv2.legacy.TrackerCSRT_create()
tracker.init(frame, bbox)

True

In [7]:
while True:
  ret,frame=cap.read()
  if ret:
    frame=cv2.rotate(frame,cv2.ROTATE_90_CLOCKWISE)
    frame=imutils.resize(frame,width=800)
    succ,bbox=tracker.update(frame)
    next=time.time()
    fps=1/(next-prev)
    prev=next
    if succ:
      x,y,w,h=[int(a) for a in bbox]
      cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
      cv2.putText(frame,"Tracking",(75,75),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
      cv2.putText(frame,"FPS: "+str(fps),(75,50),cv2.FONT_HERSHEY_SIMPLEX,0.7,(0,255,0),2)
    cv2.imshow("Frame",frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
            break
cap.release()
cv2.destroyAllWindows()

In [7]:
cap = cv2.VideoCapture(1)

ret, frame = cap.read()

if not ret:
    print("Camera error")
    cap.release()
    exit()

frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
frame = imutils.resize(frame, width=800)

cv2.imshow("Frame", frame)

bbox = cv2.selectROI("Frame", frame)

tracker.init(frame, bbox)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    frame = imutils.resize(frame, width=800)

    success, bbox = tracker.update(frame)

    if success:

        x, y, w, h = [int(a) for a in bbox]

        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.putText(frame, "Tracking",
                    (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 255, 0),
                    2)

    else:

        cv2.putText(frame, "Tracking Lost",
                    (30, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 0, 255),
                    2)

    cv2.imshow("Frame", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import imutils

cap = cv2.VideoCapture(0)

ret, frame = cap.read()

if not ret:
    print("Camera error")
    cap.release()
    exit()

frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
frame = imutils.resize(frame, width=800)

bbox = cv2.selectROI("Frame", frame)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Camera error")
        break

    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    frame = imutils.resize(frame, width=800)

    cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])),
                  (int(bbox[0]+bbox[2]), int(bbox[1]+bbox[3])), (0,255,0), 2)

    cv2.imshow("Frame", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import imutils

cap = cv2.VideoCapture(0)

tracker = None
tracking = False

drawing = False
start_point = None
end_point = None
bbox = None


def mouse_callback(event, x, y, flags, param):
    global drawing, start_point, end_point, bbox

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        start_point = (x, y)
        end_point = (x, y)

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        end_point = (x, y)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        end_point = (x, y)

        x1 = min(start_point[0], end_point[0])
        y1 = min(start_point[1], end_point[1])
        x2 = max(start_point[0], end_point[0])
        y2 = max(start_point[1], end_point[1])

        bbox = (x1, y1, x2 - x1, y2 - y1)


cv2.namedWindow("Frame")
cv2.setMouseCallback("Frame", mouse_callback)


while True:

    ret, frame = cap.read()

    if not ret:
        print("Camera error")
        break

    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
    frame = imutils.resize(frame, width=800)

    # Draw ROI while selecting
    if drawing and start_point is not None and end_point is not None:
        cv2.rectangle(frame, start_point, end_point, (0, 255, 255), 2)

        cv2.putText(frame, "Selecting...",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 255), 2)

    # Initialize tracker after ROI selection
    if bbox is not None and not drawing and not tracking:

        x, y, w, h = bbox

        if w > 0 and h > 0:

            tracker = cv2.legacy.TrackerCSRT_create()
            tracker.init(frame, bbox)

            tracking = True

            print("Tracker initialized")

        bbox = None

    # Tracking
    if tracking:

        success, tracked_bbox = tracker.update(frame)

        if success:

            x, y, w, h = [int(v) for v in tracked_bbox]

            cv2.rectangle(frame,
                          (x, y),
                          (x + w, y + h),
                          (0, 255, 0), 2)

            cv2.putText(frame,
                        "TRACKING",
                        (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 255, 0), 2)

        else:

            tracking = False
            tracker = None

            cv2.putText(frame,
                        "TRACKING LOST - SELECT OBJECT",
                        (20, 40),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0, 0, 255), 2)

            print("Tracking lost - select object again")

    # Instructions when not tracking
    if not tracking and not drawing:

        cv2.putText(frame,
                    "DRAG MOUSE TO SELECT OBJECT",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 255, 255), 2)

        cv2.putText(frame,
                    "Press R to reselect | Q to quit",
                    (20, 70),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (255, 255, 255), 2)

    cv2.imshow("Frame", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("q"):
        break

    if key == ord("r"):

        tracking = False
        tracker = None
        bbox = None
        drawing = False
        start_point = None
        end_point = None

        print("Ready to select a new object")


cap.release()
cv2.destroyAllWindows()